# Coastal Watch AI — Prototype

**UN SDG 14: Life Below Water**

This notebook adapts **Lab 3 (Image Recognition)** for monitoring illegal fishing in California Marine Protected Areas — specifically the Monterey Bay National Marine Sanctuary.

**What this prototype does**
1. Takes a coastal photo as input.
2. Asks Gemini four questions about what it sees: vessel, gear, urgency, recommended action.
3. Returns a structured "poaching probability" assessment that could be sent to a ranger's phone.

**What it does not do.** It does not make automated decisions. Every output is a recommendation for a human ranger to review.

**Team:** Mindy Baker, Anahy Diaz, Antonio Lopez, Francisco Rivas

> **First time in Colab?** Run cells top to bottom (▶ button or `Shift + Enter`). If something breaks, do `Runtime → Restart and run all`.


## Setup

Run the cell below once to install the libraries. Same as Lab 3.


In [ ]:
!pip install -q google-generativeai Pillow


## Initialize Gemini

Same as Lab 3 — store your Gemini API key in **Colab Secrets** as `GEMINI_API_KEY`.

1. Click the key icon in the left sidebar.
2. Add a secret named `GEMINI_API_KEY`.
3. Paste your key.
4. Toggle **Notebook access** ON.

Free key from <https://aistudio.google.com>.


In [ ]:
import google.generativeai as genai
from google.colab import userdata, files
from IPython.display import display
from PIL import Image as PILImage
import time
import json

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
print("Gemini initialized — ready for Coastal Watch.")


## Step 1: Upload a coastal photo

Upload an image of a coastline, harbor, or open water. Good options:

- A photo from your own visit to Monterey, Point Lobos, or any California coast.
- A still frame from a public shore-camera live stream (e.g., explore.org).
- Any creative-commons coastal photo, with or without a vessel.

Try at least three different images so you can see where the AI is reliable and where it isn't:
1. A vessel that's clearly fishing (rods, nets, lines visible).
2. A vessel that looks similar but isn't fishing (kayaker, sailboat, research craft).
3. An empty stretch of water — no vessel at all.

Supported formats: jpg, jpeg, png, webp. Max 20 MB.


In [ ]:
uploaded = files.upload()
image_filename = list(uploaded.keys())[0]

img = PILImage.open(image_filename)
print(f"Uploaded: {image_filename}")
print(f"Image size: {img.size[0]}x{img.size[1]} pixels")
display(img)


## Step 2: Define the analysis function

Same `analyze_image` helper from Lab 3. It sends the image plus a question to Gemini and returns the response. The 12-second sleep keeps us under the free-tier rate limit — please don't remove it.


In [ ]:
def analyze_image(image_path, question):
    """Send image + question to Gemini. Returns (response_text, usage_metadata)."""
    m = genai.GenerativeModel(model_name="gemini-2.5-flash")
    img = PILImage.open(image_path)
    response = m.generate_content([question, img])
    time.sleep(12)  # stay under free-tier rate limit
    return response.text, response.usage_metadata

print("analyze_image() ready.")


## Step 3: Four MPA analysis questions

These are the marine equivalent of Lab 3's four civic questions. Each one extracts a piece of information a ranger would need before deciding to drive out.

| # | Field | What it answers |
|---|-------|-----------------|
| 1 | VESSEL | Is there a boat in the image, and what kind? |
| 2 | GEAR | Is fishing gear visible — lines, nets, traps, rods? |
| 3 | URGENCY | How likely is this a poaching event? |
| 4 | ACTION | Should a ranger respond, and what should the alert say? |

Running all four may take up to a minute because of the rate-limit sleeps. That is expected.


In [ ]:
mpa_questions = [
    ("VESSEL",  "Look at this image of a coastal area. Is there a boat or vessel visible? "
                "If yes, describe it — type (kayak, fishing boat, sailboat, jet ski, etc.), "
                "approximate size, and what it appears to be doing. "
                "If no vessel is visible, say so clearly."),
    ("GEAR",    "Look at this image. Is any fishing gear visible — fishing lines, nets, traps, "
                "buoys, or rods? If yes, describe what you see and where it is in the frame. "
                "If no fishing gear is visible, say so clearly."),
    ("URGENCY", "Imagine this photo was taken inside a no-take Marine Protected Area where "
                "commercial fishing is prohibited. Based ONLY on what you can see, rate the "
                "poaching probability as LOW, MEDIUM, or HIGH. Explain your rating in 2 sentences."),
    ("ACTION",  "If a park ranger received this photo as an alert, should they investigate? "
                "Recommend YES or NO. Then in one sentence, write the text of the alert "
                "message you would send to the ranger's phone (under 160 characters)."),
]

mpa_results = {"answers": {}, "total_tokens": 0}

for label, question in mpa_questions:
    print(f"--- {label} ---")
    answer, usage = analyze_image(image_filename, question)
    mpa_results["answers"][label] = answer
    mpa_results["total_tokens"] += usage.total_token_count
    print(answer)
    print()

print(f"--- Total tokens used: {mpa_results['total_tokens']} ---")
print("Running on Gemini free tier — no cost.")


## Step 4: Single structured assessment

The four answers above are useful for a person reading them. A ranger's phone needs the **same shape every time** — predictable fields the alert system can trust.

The cell below asks Gemini for one combined assessment in a fixed JSON format. This is the message that would actually go to the ranger in production.


In [ ]:
structured_prompt = """
You are an automated marine protected area monitoring assistant.

Look at this photo, taken from a shore camera at a California Marine Protected Area
where commercial fishing is prohibited. Analyze it and return ONLY a JSON object with
this exact shape, no extra text or commentary:

{
  "vessel_present": true or false,
  "vessel_type": "string (e.g., 'kayak', 'fishing boat', 'sailboat', 'none')",
  "fishing_gear_visible": true or false,
  "gear_description": "string or 'none'",
  "poaching_probability": "LOW" | "MEDIUM" | "HIGH",
  "reason": "one sentence explaining the score",
  "ranger_alert": "string under 160 characters — the message to send"
}
""".strip()

structured_text, usage = analyze_image(image_filename, structured_prompt)

print("Raw response:")
print(structured_text)
print(f"\nTokens used: {usage.total_token_count}")

# Try to parse the response as JSON. Gemini sometimes wraps JSON in ``` fences.
try:
    cleaned = structured_text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("```")[1]
        if cleaned.lower().startswith("json"):
            cleaned = cleaned[4:].strip()
    parsed = json.loads(cleaned)
    print("\nParsed assessment:")
    for k, v in parsed.items():
        print(f"  {k}: {v}")
except Exception as e:
    print(f"\nCould not parse JSON ({e}). Inspect the raw response above and adjust the prompt.")


## Step 5: Ask your own question

Use this cell to test edge cases for your **Part 4 (Ethics)** writeup. Some good probes:

- "Could this image be confused with a research vessel?"
- "Is this boat inside or outside the protected area? How can you tell?"
- "What time of day is this? Could lighting affect your analysis?"
- "What would change about your answer if visibility were poor?"

The point of this section is to deliberately stress-test the model and document where it breaks.


In [ ]:
custom_question = input("\nAsk your own question about the image: ")
custom_result, custom_usage = analyze_image(image_filename, custom_question)

print("\n--- Response ---")
print(custom_result)
print(f"\nTokens used: {custom_usage.total_token_count}")


## Step 6: Reflection — what to write up

Use these prompts to draft your README's **Failure Case** and **Oversight** sections (Part 4 of the project).

---

**Question 1 — What did the AI get right?**
Look at the four answers above. Where did the AI's reading match what you actually see in the photo? Be specific — name the field and the answer.

*Your answer here:*

---

**Question 2 — What did the AI get wrong (or could plausibly get wrong)?**
Pick one specific moment where the AI's answer would be misleading if a ranger acted on it without checking. Describe the input image, the output Gemini gave, and what the real-world cost would be (whose time, whose money, whose trust).

*Your answer here:*

---

**Question 3 — Where does human review have to sit?**
Given what you just observed, at what point in the workflow does a human have to step in before the AI's output causes harm? In one sentence, name the trigger (e.g., "any alert with HIGH probability and a non-empty `gear_description`").

*Your answer here:*

---

**Question 4 — The one change.**
What is one change to the prototype that would reduce the harm you named in Q2? State the cost honestly — in ranger time, AI calls, or reduced automation.

*Your answer here:*
